# Natural Language → OR-Tools Constraints Scheduler

This Google Colab notebook extends your existing OR-Tools CP-SAT scheduling code with a free/open-source Hugging Face model.

It lets you write requests like:

> Schedule at least 4 visits per day, high customers should be visited 3 to 5 times, and avoid March 10 and March 25 in South Zone.

The notebook converts that text into structured constraints and applies them before running your scheduler.

## 1. Install dependencies

Run this in Colab. The default model is `google/flan-t5-base`. For faster CPU-only testing, change it to `google/flan-t5-small`.

In [1]:
!pip -q install ortools pandas "transformers<5" sentencepiece accelerate


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 14.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.


## 2. Imports

In [2]:
import ast
import calendar
import json
import re
from dataclasses import dataclass, field
from typing import Any

import pandas as pd
from ortools.sat.python import cp_model

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

## 3. Your OR-Tools scheduler class

This is your original scheduler class, kept mostly unchanged.

In [3]:
import ast
import calendar
from dataclasses import dataclass
from typing import Any

import pandas as pd
from ortools.sat.python import cp_model


@dataclass
class ScheduleResult:
    detailed_schedule: pd.DataFrame
    daily_schedule: pd.DataFrame
    selected_customers: pd.DataFrame


class ConstraintScheduling:
    def __init__(
        self,
        priority_weights: dict[str, int] | None = None,
        preference_weights: dict[str, int] | None = None,
        min_daily_visits: int = 3,
        extra_high_bonus: int = 25,
    ) -> None:
        self.priority_weights = priority_weights or {
            "High": 100,
            "Medium": 50,
            "Low": 10,
        }
        self.preference_weights = preference_weights or {
            "day_and_week": 20,
            "day_only": 10,
            "week_only": 5,
            "other": 0,
        }
        self.min_daily_visits = min_daily_visits
        self.extra_high_bonus = extra_high_bonus

    def create_daily_schedule(
        self,
        rfm_df: pd.DataFrame,
        territory_df: pd.DataFrame,
        best_visit_df: pd.DataFrame,
        month_start_date: str | pd.Timestamp,
    ) -> ScheduleResult:
        month_start_date = pd.Timestamp(month_start_date).normalize()

        customers = self._build_customer_master(rfm_df, best_visit_df)
        territory_master = self._prepare_territory_master(territory_df)

        selected_customers = self._filter_customers_by_territory_capacity(
            customers=customers,
            territory_master=territory_master,
        )

        detailed_parts: list[pd.DataFrame] = []

        for territory_name, territory_customers in selected_customers.groupby("territory_name"):
            territory_row = territory_master.loc[
                territory_master["territory_name"] == territory_name
            ].iloc[0]

            valid_dates = self._get_valid_dates_for_territory(
                month_start_date=month_start_date,
                holiday_list=territory_row["holiday_list"],
            )

            if not valid_dates:
                continue

            territory_schedule = self._solve_territory_schedule(
                customers=territory_customers.reset_index(drop=True),
                territory_row=territory_row,
                valid_dates=valid_dates,
            )

            if not territory_schedule.empty:
                detailed_parts.append(territory_schedule)

        detailed_schedule = (
            pd.concat(detailed_parts, ignore_index=True)
            if detailed_parts
            else pd.DataFrame(
                columns=[
                    "schedule_date",
                    "territory_name",
                    "customer_id",
                    "shop_name",
                    "shop_location",
                    "latitude",
                    "longitude",
                    "rfm_segment_final",
                    "rfm_score",
                ]
            )
        )

        daily_schedule = self._build_daily_customer_list(detailed_schedule)

        return ScheduleResult(
            detailed_schedule=detailed_schedule,
            daily_schedule=daily_schedule,
            selected_customers=selected_customers,
        )

    def _build_customer_master(
        self,
        rfm_df: pd.DataFrame,
        best_visit_df: pd.DataFrame,
    ) -> pd.DataFrame:
        required_rfm_cols = [
            "customer_id",
            "shop_name",
            "shop_location",
            "territory_name",
            "latitude",
            "longitude",
            "rfm_score",
            "rfm_segment_final",
            "recency",
            "frequency",
            "monetary",
        ]

        rfm_cols = [c for c in required_rfm_cols if c in rfm_df.columns]
        best_cols = [c for c in ["customer_id", "best_days", "best_weeks"] if c in best_visit_df.columns]

        customers = rfm_df[rfm_cols].drop_duplicates("customer_id").merge(
            best_visit_df[best_cols].drop_duplicates("customer_id"),
            on="customer_id",
            how="left",
        )

        customers["best_days"] = customers["best_days"].apply(self._ensure_list)
        customers["best_weeks"] = customers["best_weeks"].apply(self._ensure_list)
        customers["rfm_segment_final"] = customers["rfm_segment_final"].fillna("Low")
        customers["rfm_score"] = customers["rfm_score"].fillna(0.0)
        customers["recency"] = customers["recency"].fillna(999)
        customers["frequency"] = customers["frequency"].fillna(0)
        customers["monetary"] = customers["monetary"].fillna(0.0)
        customers["min_visits"] = customers["rfm_segment_final"].apply(self._get_min_visits)
        customers["max_visits"] = customers["rfm_segment_final"].apply(self._get_max_visits)

        return customers

    def _prepare_territory_master(self, territory_df: pd.DataFrame) -> pd.DataFrame:
        territory_master = territory_df.copy()
        territory_master["holiday_list"] = territory_master["holiday_list"].apply(self._ensure_date_list)
        return territory_master

    def _filter_customers_by_territory_capacity(
        self,
        customers: pd.DataFrame,
        territory_master: pd.DataFrame,
    ) -> pd.DataFrame:
        selected_parts: list[pd.DataFrame] = []

        for territory_name, group in customers.groupby("territory_name"):
            territory_row = territory_master.loc[
                territory_master["territory_name"] == territory_name
            ].iloc[0]

            max_customers = int(territory_row["max_customers_capacity"])
            ranked_group = self._rank_customers(group)
            selected_parts.append(ranked_group.head(max_customers))

        if not selected_parts:
            return customers.iloc[0:0].copy()

        selected_customers = pd.concat(selected_parts, ignore_index=True)
        return selected_customers.drop(columns=["segment_priority"], errors="ignore")

    def _rank_customers(self, customers: pd.DataFrame) -> pd.DataFrame:
        ranked = customers.copy()
        segment_priority_map = {"High": 1, "Medium": 2, "Low": 3}
        ranked["segment_priority"] = ranked["rfm_segment_final"].map(segment_priority_map).fillna(3)

        ranked = ranked.sort_values(
            by=["segment_priority", "rfm_score", "recency", "monetary", "frequency"],
            ascending=[True, False, True, False, False],
        )

        return ranked

    def _get_valid_dates_for_territory(
        self,
        month_start_date: pd.Timestamp,
        holiday_list: list[pd.Timestamp],
    ) -> list[pd.Timestamp]:
        year = month_start_date.year
        month = month_start_date.month
        days_in_month = calendar.monthrange(year, month)[1]

        all_dates = pd.date_range(start=month_start_date, periods=days_in_month, freq="D")
        holiday_set = {pd.Timestamp(d).normalize() for d in holiday_list}

        valid_dates = [
            d.normalize()
            for d in all_dates
            if d.normalize() not in holiday_set
        ]

        return valid_dates

    def _solve_territory_schedule(
        self,
        customers: pd.DataFrame,
        territory_row: pd.Series,
        valid_dates: list[pd.Timestamp],
    ) -> pd.DataFrame:
        model = cp_model.CpModel()

        customer_ids = customers["customer_id"].tolist()
        territory_name = territory_row["territory_name"]

        empty_result = pd.DataFrame(
            columns=[
                "schedule_date",
                "territory_name",
                "customer_id",
                "shop_name",
                "shop_location",
                "latitude",
                "longitude",
                "rfm_segment_final",
                "rfm_score",
            ]
        )

        if not customer_ids or not valid_dates:
            return empty_result

        x: dict[tuple[str, pd.Timestamp], cp_model.IntVar] = {}

        for _, customer in customers.iterrows():
            customer_id = customer["customer_id"]
            for visit_date in valid_dates:
                x[(customer_id, visit_date)] = model.NewBoolVar(
                    f"x_{customer_id}_{visit_date.strftime('%Y%m%d')}"
                )

        for _, customer in customers.iterrows():
            customer_id = customer["customer_id"]
            customer_vars = [x[(customer_id, d)] for d in valid_dates]
            model.Add(sum(customer_vars) >= int(customer["min_visits"]))
            model.Add(sum(customer_vars) <= int(customer["max_visits"]))

        territory_daily_capacity = int(territory_row["daily_visit_capacity"])
        monthly_capacity = int(territory_row["monthly_visit_capacity"])

        total_customer_min_visits = int(customers["min_visits"].sum())
        total_customer_max_visits = int(customers["max_visits"].sum())
        total_possible_visits = min(monthly_capacity, total_customer_max_visits)

        if total_customer_min_visits > monthly_capacity:
            raise ValueError(
                f"Infeasible schedule for {territory_name}: "
                f"minimum customer visits={total_customer_min_visits} exceeds "
                f"monthly_visit_capacity={monthly_capacity}"
            )

        feasible_min_daily = min(
            self.min_daily_visits,
            territory_daily_capacity,
            total_possible_visits // len(valid_dates),
        )

        for visit_date in valid_dates:
            day_vars = [x[(customer_id, visit_date)] for customer_id in customer_ids]
            model.Add(sum(day_vars) >= feasible_min_daily)
            model.Add(sum(day_vars) <= territory_daily_capacity)

        all_vars = [x[(customer_id, d)] for customer_id in customer_ids for d in valid_dates]
        model.Add(sum(all_vars) <= monthly_capacity)

        objective_terms = []

        for _, customer in customers.iterrows():
            customer_id = customer["customer_id"]
            rfm_segment = customer["rfm_segment_final"]
            best_days = set(customer["best_days"])
            best_weeks = set(customer["best_weeks"])

            base_weight = self.priority_weights.get(rfm_segment, 0)
            high_bonus = self.extra_high_bonus if rfm_segment == "High" else 0

            for visit_date in valid_dates:
                pref_weight = self._get_preference_weight(
                    visit_date=visit_date,
                    best_days=best_days,
                    best_weeks=best_weeks,
                )
                total_weight = base_weight + pref_weight + high_bonus
                objective_terms.append(total_weight * x[(customer_id, visit_date)])

        model.Maximize(sum(objective_terms))

        solver = cp_model.CpSolver()
        solver.parameters.max_time_in_seconds = 30
        solver.parameters.num_search_workers = 8

        status = solver.Solve(model)

        if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
            return empty_result

        rows: list[dict[str, Any]] = []
        customer_lookup = customers.set_index("customer_id").to_dict("index")

        for customer_id in customer_ids:
            for visit_date in valid_dates:
                if solver.Value(x[(customer_id, visit_date)]) == 1:
                    info = customer_lookup[customer_id]
                    rows.append(
                        {
                            "schedule_date": visit_date,
                            "territory_name": territory_name,
                            "customer_id": customer_id,
                            "shop_name": info["shop_name"],
                            "shop_location": info["shop_location"],
                            "latitude": info.get("latitude"),
                            "longitude": info.get("longitude"),
                            "rfm_segment_final": info["rfm_segment_final"],
                            "rfm_score": info["rfm_score"],
                        }
                    )

        result = pd.DataFrame(rows).sort_values(
            ["schedule_date", "customer_id"]
        ).reset_index(drop=True)

        return result

    def _build_daily_customer_list(self, detailed_schedule: pd.DataFrame) -> pd.DataFrame:
        if detailed_schedule.empty:
            return pd.DataFrame(
                columns=[
                    "schedule_date",
                    "territory_name",
                    "customer_list",
                    "customer_count",
                ]
            )

        daily_schedule = (
            detailed_schedule.groupby(["schedule_date", "territory_name"])
            .agg(
                customer_list=("customer_id", list),
                customer_count=("customer_id", "nunique"),
            )
            .reset_index()
            .sort_values(["schedule_date", "territory_name"])
            .reset_index(drop=True)
        )

        return daily_schedule

    def _get_min_visits(self, rfm_segment: str) -> int:
        if rfm_segment == "High":
            return 2
        return 1

    def _get_max_visits(self, rfm_segment: str) -> int:
        if rfm_segment == "High":
            return 4
        if rfm_segment == "Medium":
            return 2
        return 1

    def _get_preference_weight(
        self,
        visit_date: pd.Timestamp,
        best_days: set[str],
        best_weeks: set[int],
    ) -> int:
        weekday_name = visit_date.day_name()
        week_of_month = ((visit_date.day - 1) // 7) + 1

        day_match = weekday_name in best_days
        week_match = week_of_month in best_weeks

        if day_match and week_match:
            return self.preference_weights["day_and_week"]
        if day_match:
            return self.preference_weights["day_only"]
        if week_match:
            return self.preference_weights["week_only"]
        return self.preference_weights["other"]

    def _ensure_list(self, value: Any) -> list[Any]:
        if isinstance(value, list):
            return value

        if value is None:
            return []

        if isinstance(value, float) and pd.isna(value):
            return []

        if isinstance(value, str):
            value = value.strip()
            if value.startswith("[") and value.endswith("]"):
                try:
                    parsed = ast.literal_eval(value)
                    return parsed if isinstance(parsed, list) else [parsed]
                except (ValueError, SyntaxError):
                    return [value]
            return [value]

        return [value]

    def _ensure_date_list(self, value: Any) -> list[pd.Timestamp]:
        raw_list = self._ensure_list(value)
        return [pd.Timestamp(v).normalize() for v in raw_list]

## 4. Constraint schema for natural language

The LLM will output JSON in this shape. We keep the schema small because it is easier to validate and safer to plug into optimization code.

In [4]:
@dataclass
class NaturalLanguageConstraints:
    min_daily_visits: int | None = None
    extra_high_bonus: int | None = None
    priority_weights: dict[str, int] = field(default_factory=dict)
    preference_weights: dict[str, int] = field(default_factory=dict)
    segment_min_visits: dict[str, int] = field(default_factory=dict)
    segment_max_visits: dict[str, int] = field(default_factory=dict)
    territory_daily_capacity: dict[str, int] = field(default_factory=dict)
    territory_monthly_capacity: dict[str, int] = field(default_factory=dict)
    territory_max_customers_capacity: dict[str, int] = field(default_factory=dict)
    add_holidays: dict[str, list[str]] = field(default_factory=dict)
    explanation: str = ""

## 5. Load a free Hugging Face text-to-text model

No OpenAI key is needed. This downloads the model in the Colab runtime. This notebook calls the model directly instead of using `pipeline("text2text-generation")`, which avoids task-registry errors in newer Transformers versions.


In [5]:
MODEL_NAME = "google/flan-t5-base"  # Try "google/flan-t5-small" for faster CPU runs.

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def generate_text(prompt: str, max_new_tokens: int = 512) -> str:
    """Generate text without using pipeline(), so this works with Transformers v4/v5 task registry changes."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## 6. Natural language parser

This function asks FLAN-T5 to return JSON only. It also includes a lightweight rule-based fallback for common scheduling phrases.

In [6]:
ALLOWED_TOP_LEVEL_KEYS = {
    "min_daily_visits",
    "extra_high_bonus",
    "priority_weights",
    "preference_weights",
    "segment_min_visits",
    "segment_max_visits",
    "territory_daily_capacity",
    "territory_monthly_capacity",
    "territory_max_customers_capacity",
    "add_holidays",
    "explanation",
}

SEGMENTS = {"High", "Medium", "Low"}


def _extract_json(text: str) -> dict[str, Any]:
    """Extract the first JSON object from model output."""
    text = text.strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return {}
    candidate = text[start : end + 1]
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        # Try simple cleanup for single quotes.
        candidate = candidate.replace("'", '"')
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            return {}


def _coerce_int_dict(value: Any) -> dict[str, int]:
    if not isinstance(value, dict):
        return {}
    out = {}
    for k, v in value.items():
        try:
            out[str(k)] = int(v)
        except Exception:
            pass
    return out


def _coerce_holiday_dict(value: Any) -> dict[str, list[str]]:
    if not isinstance(value, dict):
        return {}
    out = {}
    for territory, dates in value.items():
        if isinstance(dates, str):
            dates = [dates]
        if isinstance(dates, list):
            clean_dates = []
            for d in dates:
                try:
                    clean_dates.append(str(pd.Timestamp(d).date()))
                except Exception:
                    pass
            if clean_dates:
                out[str(territory)] = clean_dates
    return out


def validate_constraints(raw: dict[str, Any]) -> NaturalLanguageConstraints:
    """Validate and coerce model JSON before using it."""
    raw = {k: v for k, v in raw.items() if k in ALLOWED_TOP_LEVEL_KEYS}

    def optional_int(key: str) -> int | None:
        if key not in raw or raw[key] is None:
            return None
        try:
            return int(raw[key])
        except Exception:
            return None

    constraints = NaturalLanguageConstraints(
        min_daily_visits=optional_int("min_daily_visits"),
        extra_high_bonus=optional_int("extra_high_bonus"),
        priority_weights=_coerce_int_dict(raw.get("priority_weights", {})),
        preference_weights=_coerce_int_dict(raw.get("preference_weights", {})),
        segment_min_visits={
            k: v for k, v in _coerce_int_dict(raw.get("segment_min_visits", {})).items()
            if k in SEGMENTS
        },
        segment_max_visits={
            k: v for k, v in _coerce_int_dict(raw.get("segment_max_visits", {})).items()
            if k in SEGMENTS
        },
        territory_daily_capacity=_coerce_int_dict(raw.get("territory_daily_capacity", {})),
        territory_monthly_capacity=_coerce_int_dict(raw.get("territory_monthly_capacity", {})),
        territory_max_customers_capacity=_coerce_int_dict(raw.get("territory_max_customers_capacity", {})),
        add_holidays=_coerce_holiday_dict(raw.get("add_holidays", {})),
        explanation=str(raw.get("explanation", "")),
    )

    return constraints


def rule_based_constraint_fallback(user_request: str) -> dict[str, Any]:
    """Catches common phrases if the model returns invalid JSON."""
    text = user_request.lower()
    out: dict[str, Any] = {}

    # Examples: "minimum 4 visits per day", "at least 4 daily visits"
    m = re.search(r"(?:minimum|at least|min)\s+(\d+)\s+(?:visits?|customers?)\s+(?:per day|daily|a day)", text)
    if m:
        out["min_daily_visits"] = int(m.group(1))

    # Examples: "high customers 3 to 5 visits", "High min 3 max 5"
    for segment in ["high", "medium", "low"]:
        m = re.search(segment + r".{0,30}?(?:min(?:imum)?\s*)?(\d+)\s*(?:to|-)\s*(\d+).{0,20}?visits?", text)
        if m:
            out.setdefault("segment_min_visits", {})[segment.title()] = int(m.group(1))
            out.setdefault("segment_max_visits", {})[segment.title()] = int(m.group(2))

    # Examples: "high customers at least 3 visits"
    for segment in ["high", "medium", "low"]:
        m = re.search(segment + r".{0,30}?(?:at least|min(?:imum)?)\s*(\d+).{0,20}?visits?", text)
        if m:
            out.setdefault("segment_min_visits", {})[segment.title()] = int(m.group(1))

    # Examples: "high customers at most 5 visits"
    for segment in ["high", "medium", "low"]:
        m = re.search(segment + r".{0,30}?(?:at most|max(?:imum)?)\s*(\d+).{0,20}?visits?", text)
        if m:
            out.setdefault("segment_max_visits", {})[segment.title()] = int(m.group(1))

    return out


def parse_natural_language_constraints(user_request: str) -> NaturalLanguageConstraints:
    prompt = f"""
Convert the scheduling request into JSON only.

Allowed JSON keys:
- min_daily_visits: integer or null
- extra_high_bonus: integer or null
- priority_weights: object with optional High, Medium, Low integer values
- preference_weights: object with optional day_and_week, day_only, week_only, other integer values
- segment_min_visits: object with optional High, Medium, Low integer values
- segment_max_visits: object with optional High, Medium, Low integer values
- territory_daily_capacity: object mapping territory name to integer
- territory_monthly_capacity: object mapping territory name to integer
- territory_max_customers_capacity: object mapping territory name to integer
- add_holidays: object mapping territory name to list of YYYY-MM-DD dates
- explanation: short string

Rules:
- Return JSON only.
- Do not invent territory names.
- If a constraint is not mentioned, omit it or set it to null.
- Use exact segment names: High, Medium, Low.

Request:
{user_request}

JSON:
""".strip()

    model_output = generate_text(prompt)
    raw = _extract_json(model_output)

    if not raw:
        raw = rule_based_constraint_fallback(user_request)
        raw["explanation"] = "Parsed with fallback rules because model output was not valid JSON."
    else:
        fallback = rule_based_constraint_fallback(user_request)
        # Fallback values fill gaps, but model values take priority.
        for key, value in fallback.items():
            if key not in raw or raw[key] in (None, {}, []):
                raw[key] = value

    return validate_constraints(raw)

## 7. Apply parsed constraints to your scheduler and input data

Some constraints belong in the `ConstraintScheduling(...)` constructor. Others update `territory_master` before solving.

In [11]:
import re
from dataclasses import dataclass, field

@dataclass
class NaturalLanguageConstraints:
    min_daily_visits: int | None = None
    extra_high_bonus: int | None = None
    priority_weights: dict = field(default_factory=dict)
    preference_weights: dict = field(default_factory=dict)
    segment_min_visits: dict = field(default_factory=dict)
    segment_max_visits: dict = field(default_factory=dict)
    territory_daily_capacity: dict = field(default_factory=dict)
    territory_monthly_capacity: dict = field(default_factory=dict)
    territory_max_customers_capacity: dict = field(default_factory=dict)
    add_holidays: dict = field(default_factory=dict)
    explanation: str = ""


def parse_natural_language_constraints(user_request: str) -> NaturalLanguageConstraints:
    text = user_request.lower().replace("\n", " ")

    constraints = NaturalLanguageConstraints()
    constraints.explanation = "Parsed with improved rule-based parser."

    # Minimum daily visits: "Set minimum daily visits to 4"
    match = re.search(
        r"(?:set\s+)?minimum\s+daily\s+visits?\s*(?:to|as|=|is)?\s*(\d+)",
        text,
    )
    if match:
        constraints.min_daily_visits = int(match.group(1))

    # Extra high bonus: "Give high priority customers extra bonus 40"
    match = re.search(
        r"(?:high\s+priority\s+customers?.*?extra\s+bonus|extra\s+high\s+bonus)\s*(?:to|as|=|is)?\s*(\d+)",
        text,
    )
    if match:
        constraints.extra_high_bonus = int(match.group(1))

    segment_map = {
        "high": "High",
        "medium": "Medium",
        "low": "Low",
    }

    for segment_lower, segment_name in segment_map.items():

        # "High customers should be visited between 3 and 5 times"
        match = re.search(
            rf"{segment_lower}\s+customers?.*?(?:between|from)\s+(\d+)\s+(?:and|to)\s+(\d+)\s+times?",
            text,
        )
        if match:
            constraints.segment_min_visits[segment_name] = int(match.group(1))
            constraints.segment_max_visits[segment_name] = int(match.group(2))

        # "Medium customers should be visited at least 2 times"
        match = re.search(
            rf"{segment_lower}\s+customers?.*?at\s+least\s+(\d+)\s+times?",
            text,
        )
        if match:
            constraints.segment_min_visits[segment_name] = int(match.group(1))

        # "Low customers should be visited at most 1 time"
        match = re.search(
            rf"{segment_lower}\s+customers?.*?at\s+most\s+(\d+)\s+times?",
            text,
        )
        if match:
            constraints.segment_max_visits[segment_name] = int(match.group(1))

    return constraints

In [12]:
user_request = """
Set minimum daily visits to 4.
High customers should be visited between 3 and 5 times.
Medium customers should be visited at least 2 times.
Give high priority customers extra bonus 40.
"""

parsed_constraints = parse_natural_language_constraints(user_request)

print("Parsed constraints:")
print(parsed_constraints)

Parsed constraints:
NaturalLanguageConstraints(min_daily_visits=4, extra_high_bonus=40, priority_weights={}, preference_weights={}, segment_min_visits={'High': 2, 'Medium': 2}, segment_max_visits={'High': 5}, territory_daily_capacity={}, territory_monthly_capacity={}, territory_max_customers_capacity={}, add_holidays={}, explanation='Parsed with improved rule-based parser.')


In [5]:
import re
from dataclasses import dataclass, field, asdict
from pprint import pprint


@dataclass
class NaturalLanguageConstraints:
    min_daily_visits: int | None = None
    extra_high_bonus: int | None = None

    priority_weights: dict = field(default_factory=dict)
    preference_weights: dict = field(default_factory=dict)

    segment_min_visits: dict = field(default_factory=dict)
    segment_max_visits: dict = field(default_factory=dict)

    territory_daily_capacity: dict = field(default_factory=dict)
    territory_monthly_capacity: dict = field(default_factory=dict)
    territory_max_customers_capacity: dict = field(default_factory=dict)

    add_holidays: dict = field(default_factory=dict)

    salesman_working_hours: dict = field(default_factory=dict)

    max_route_distance_km: float | None = None
    max_travel_time_minutes: int | None = None

    group_nearby_customers: bool = False
    nearby_customer_radius_km: float | None = None

    warnings: list = field(default_factory=list)
    explanation: str = ""


def split_sentences(text: str) -> list[str]:
    text = text.replace("\n", ". ")
    sentences = re.split(r"[.;]+", text)
    return [s.strip().lower() for s in sentences if s.strip()]


def to_24h(hour, minute, ampm):
    hour = int(hour)
    minute = int(minute or 0)

    if ampm == "pm" and hour != 12:
        hour += 12

    if ampm == "am" and hour == 12:
        hour = 0

    return f"{hour:02d}:{minute:02d}"


def parse_natural_language_constraints(user_request: str) -> NaturalLanguageConstraints:

    sentences = split_sentences(user_request)
    full_text = " ".join(sentences)

    constraints = NaturalLanguageConstraints()
    constraints.explanation = "Parsed with sentence-based rule parser."

    segment_map = {
        "high": "High",
        "medium": "Medium",
        "low": "Low",
    }

    for sentence in sentences:

        match = re.search(
            r"(?:set\s+)?minimum\s+daily\s+visits?\s*(?:to|as|=|is|should\s+be)?\s*(\d+)",
            sentence,
        )
        if match:
            constraints.min_daily_visits = int(match.group(1))

        match = re.search(
            r"(?:daily\s+capacity|daily\s+visit\s+capacity|max(?:imum)?\s+daily\s+visits?)\s*(?:to|as|=|is|should\s+be)?\s*(\d+)",
            sentence,
        )
        if match:
            constraints.territory_daily_capacity["ALL"] = int(match.group(1))

        match = re.search(
            r"(?:monthly\s+capacity|monthly\s+visit\s+capacity|max(?:imum)?\s+monthly\s+visits?)\s*(?:to|as|=|is|should\s+be)?\s*(\d+)",
            sentence,
        )
        if match:
            constraints.territory_monthly_capacity["ALL"] = int(match.group(1))

        match = re.search(
            r"(?:maximum\s+customers|max\s+customers?|customer\s+capacity|max(?:imum)?\s+customer\s+capacity)\s*(?:to|as|=|is|should\s+be)?\s*(\d+)",
            sentence,
        )
        if match:
            constraints.territory_max_customers_capacity["ALL"] = int(match.group(1))

        match = re.search(
            r"(?:high\s+priority\s+customers?.*?extra\s+bonus|extra\s+high\s+bonus)\s*(?:to|as|=|is|should\s+be)?\s*(\d+)",
            sentence,
        )
        if match:
            constraints.extra_high_bonus = int(match.group(1))

        for segment_lower, segment_name in segment_map.items():

            if segment_lower not in sentence:
                continue

            match = re.search(
                rf"{segment_lower}\s+customers?.*?(?:between|from)\s+(\d+)\s+(?:and|to)\s+(\d+)\s+times?",
                sentence,
            )
            if match:
                constraints.segment_min_visits[segment_name] = int(match.group(1))
                constraints.segment_max_visits[segment_name] = int(match.group(2))
                continue

            match = re.search(
                rf"{segment_lower}\s+customers?.*?at\s+least\s+(\d+)\s+times?",
                sentence,
            )
            if match:
                constraints.segment_min_visits[segment_name] = int(match.group(1))

            match = re.search(
                rf"{segment_lower}\s+customers?.*?at\s+most\s+(\d+)\s+times?",
                sentence,
            )
            if match:
                constraints.segment_max_visits[segment_name] = int(match.group(1))

            match = re.search(
                rf"{segment_lower}\s+customers?.*?(?:exactly|only)\s+(\d+)\s+times?",
                sentence,
            )
            if match:
                value = int(match.group(1))
                constraints.segment_min_visits[segment_name] = value
                constraints.segment_max_visits[segment_name] = value

        holiday_matches = re.findall(
            r"(?:holiday|off|closed)\s+(?:on\s+)?(\d{4}-\d{2}-\d{2})",
            sentence,
        )
        if holiday_matches:
            constraints.add_holidays.setdefault("ALL", [])
            constraints.add_holidays["ALL"].extend(holiday_matches)

        match = re.search(
            r"(?:working\s+hours?|salesman\s+working\s+time|work\s+time)\s*(?:from\s+)?(\d{1,2})(?::(\d{2}))?\s*(am|pm)?\s*(?:to|-)\s*(\d{1,2})(?::(\d{2}))?\s*(am|pm)?",
            sentence,
        )
        if match:
            constraints.salesman_working_hours = {
                "start_time": to_24h(match.group(1), match.group(2), match.group(3)),
                "end_time": to_24h(match.group(4), match.group(5), match.group(6)),
            }

        match = re.search(
            r"(?:maximum\s+route\s+distance|max\s+route\s+distance|route\s+distance)\s*(?:to|as|=|is|should\s+be)?\s*(\d+(?:\.\d+)?)\s*km",
            sentence,
        )
        if match:
            constraints.max_route_distance_km = float(match.group(1))

        match = re.search(
            r"(?:maximum\s+travel\s+time|max\s+travel\s+time|travel\s+time)\s*(?:to|as|=|is|should\s+be)?\s*(\d+)\s*(?:minutes|mins|min)",
            sentence,
        )
        if match:
            constraints.max_travel_time_minutes = int(match.group(1))

        if (
            "nearby customers" in sentence
            or "close customers" in sentence
            or "closely located customers" in sentence
            or "same route" in sentence
            or "single route" in sentence
        ):
            constraints.group_nearby_customers = True

        match = re.search(
            r"(?:nearby|close|closely located).*?within\s+(\d+(?:\.\d+)?)\s*km",
            sentence,
        )
        if match:
            constraints.group_nearby_customers = True
            constraints.nearby_customer_radius_km = float(match.group(1))

    constraints.add_holidays = {
        k: sorted(list(set(v))) for k, v in constraints.add_holidays.items()
    }

    for segment in set(
        list(constraints.segment_min_visits.keys())
        + list(constraints.segment_max_visits.keys())
    ):
        min_v = constraints.segment_min_visits.get(segment)
        max_v = constraints.segment_max_visits.get(segment)

        if min_v is not None and max_v is not None and min_v > max_v:
            constraints.warnings.append(
                f"{segment} has min_visits greater than max_visits."
            )

    if (
        constraints.min_daily_visits is not None
        and constraints.territory_daily_capacity.get("ALL") is not None
        and constraints.min_daily_visits > constraints.territory_daily_capacity["ALL"]
    ):
        constraints.warnings.append(
            "min_daily_visits is greater than daily capacity."
        )

    if (
        constraints.max_route_distance_km is not None
        and constraints.max_route_distance_km <= 0
    ):
        constraints.warnings.append(
            "max_route_distance_km should be greater than 0."
        )

    if (
        constraints.max_travel_time_minutes is not None
        and constraints.max_travel_time_minutes <= 0
    ):
        constraints.warnings.append(
            "max_travel_time_minutes should be greater than 0."
        )

    return constraints


test_requests = [
    """
    Set minimum daily visits to 4.
    High customers should be visited between 3 and 5 times.
    Medium customers should be visited at least 2 times.
    Give high priority customers extra bonus 40.
    """,

    """
    Daily capacity should be 8.
    Monthly capacity should be 150.
    Maximum customers should be 80.
    """,

    """
    Holiday on 2026-03-14.
    Holiday on 2026-03-25.
    Salesman working time from 9am to 6pm.
    """,

    """
    Maximum route distance 35 km.
    Maximum travel time 90 minutes.
    Closely located customers within 3 km should be in a single route.
    """,

    """
    Low customers should be visited at most 1 time.
    Medium customers should be visited between 2 and 3 times.
    High customers should be visited between 4 and 6 times.
    """,

    """
    Set minimum daily visits to 10.
    Daily capacity should be 6.
    High customers should be visited between 5 and 3 times.
    """,

    """
    Low customers should be visited exactly 1 time.
    Medium customers should be visited exactly 2 times.
    High customers should be visited exactly 4 times.
    """,

    """
    Salesman working time from 09:30am to 06:15pm.
    Holiday on 2026-03-10.
    Holiday on 2026-03-10.
    Maximum route distance should be 42.5 km.
    """
]


for i, request in enumerate(test_requests, start=1):
    print("=" * 100)
    print(f"TEST {i}")
    print("=" * 100)
    print(request.strip())

    parsed = parse_natural_language_constraints(request)

    print("\nPARSED CONSTRAINTS:")
    pprint(asdict(parsed))

    print("\n")

TEST 1
Set minimum daily visits to 4.
    High customers should be visited between 3 and 5 times.
    Medium customers should be visited at least 2 times.
    Give high priority customers extra bonus 40.

PARSED CONSTRAINTS:
{'add_holidays': {},
 'explanation': 'Parsed with sentence-based rule parser.',
 'extra_high_bonus': 40,
 'group_nearby_customers': False,
 'max_route_distance_km': None,
 'max_travel_time_minutes': None,
 'min_daily_visits': 4,
 'nearby_customer_radius_km': None,
 'preference_weights': {},
 'priority_weights': {},
 'salesman_working_hours': {},
 'segment_max_visits': {'High': 5},
 'segment_min_visits': {'High': 3, 'Medium': 2},
 'territory_daily_capacity': {},
 'territory_max_customers_capacity': {},
 'territory_monthly_capacity': {},
 'warnings': []}


TEST 2
Daily capacity should be 8.
    Monthly capacity should be 150.
    Maximum customers should be 80.

PARSED CONSTRAINTS:
{'add_holidays': {},
 'explanation': 'Parsed with sentence-based rule parser.',
 'extr

In [6]:
import re
from dataclasses import dataclass, field, asdict
from pprint import pprint


@dataclass
class NaturalLanguageConstraints:
    min_daily_visits: int | None = None
    extra_high_bonus: int | None = None

    priority_weights: dict = field(default_factory=dict)
    preference_weights: dict = field(default_factory=dict)

    segment_min_visits: dict = field(default_factory=dict)
    segment_max_visits: dict = field(default_factory=dict)

    territory_daily_capacity: dict = field(default_factory=dict)
    territory_monthly_capacity: dict = field(default_factory=dict)
    territory_max_customers_capacity: dict = field(default_factory=dict)

    add_holidays: dict = field(default_factory=dict)

    salesman_working_hours: dict = field(default_factory=dict)

    max_route_distance_km: float | None = None
    max_travel_time_minutes: int | None = None

    group_nearby_customers: bool = False
    nearby_customer_radius_km: float | None = None

    warnings: list = field(default_factory=list)
    explanation: str = ""


SEGMENT_MAP = {
    "high": "High",
    "medium": "Medium",
    "low": "Low",
}


def split_sentences(text: str) -> list[str]:
    text = text.replace("\n", ". ")
    sentences = re.split(r"[.;]+", text)
    return [s.strip().lower() for s in sentences if s.strip()]


def first_number(sentence: str) -> int | None:
    match = re.search(r"\d+", sentence)
    return int(match.group()) if match else None


def first_float(sentence: str) -> float | None:
    match = re.search(r"\d+(?:\.\d+)?", sentence)
    return float(match.group()) if match else None


def to_24h(hour, minute=None, ampm=None) -> str:
    hour = int(hour)
    minute = int(minute or 0)

    if ampm == "pm" and hour != 12:
        hour += 12

    if ampm == "am" and hour == 12:
        hour = 0

    return f"{hour:02d}:{minute:02d}"


def set_if_found(patterns, sentence, setter):
    for pattern in patterns:
        match = re.search(pattern, sentence)
        if match:
            setter(match)
            return True
    return False


def parse_basic_capacity(sentence, constraints):
    set_if_found(
        [
            r"(?:minimum|min)\s+daily\s+visits?.*?(\d+)",
            r"(?:at\s+least)\s+(\d+)\s+visits?\s+per\s+day",
            r"(?:do|schedule)\s+minimum\s+(\d+)\s+visits?\s+daily",
        ],
        sentence,
        lambda m: setattr(constraints, "min_daily_visits", int(m.group(1))),
    )

    set_if_found(
        [
            r"(?:daily\s+capacity|daily\s+visit\s+capacity).*?(\d+)",
            r"(?:maximum|max)\s+(\d+)\s+visits?\s+per\s+day",
            r"(?:no\s+more\s+than)\s+(\d+)\s+visits?\s+per\s+day",
            r"(?:cap\s+daily\s+visits?\s+at)\s+(\d+)",
        ],
        sentence,
        lambda m: constraints.territory_daily_capacity.update({"ALL": int(m.group(1))}),
    )

    set_if_found(
        [
            r"(?:monthly\s+capacity|monthly\s+visit\s+capacity).*?(\d+)",
            r"(?:maximum|max)\s+(\d+)\s+visits?\s+per\s+month",
            r"(?:no\s+more\s+than)\s+(\d+)\s+visits?\s+per\s+month",
            r"(?:cap\s+monthly\s+visits?\s+at)\s+(\d+)",
        ],
        sentence,
        lambda m: constraints.territory_monthly_capacity.update({"ALL": int(m.group(1))}),
    )

    set_if_found(
        [
            r"(?:maximum|max)\s+customers?.*?(\d+)",
            r"(?:customer\s+capacity).*?(\d+)",
            r"(?:select\s+only)\s+(\d+)\s+customers?",
            r"(?:limit\s+customers?\s+to)\s+(\d+)",
        ],
        sentence,
        lambda m: constraints.territory_max_customers_capacity.update({"ALL": int(m.group(1))}),
    )


def parse_segment_rules(sentence, constraints):
    for segment_lower, segment_name in SEGMENT_MAP.items():

        if segment_lower not in sentence:
            continue

        if set_if_found(
            [
                rf"{segment_lower}\s+customers?.*?(?:between|from)\s+(\d+)\s+(?:and|to)\s+(\d+)",
                rf"{segment_lower}\s+customers?.*?(\d+)\s*-\s*(\d+)\s+visits?",
            ],
            sentence,
            lambda m: (
                constraints.segment_min_visits.update({segment_name: int(m.group(1))}),
                constraints.segment_max_visits.update({segment_name: int(m.group(2))}),
            ),
        ):
            continue

        set_if_found(
            [
                rf"{segment_lower}\s+customers?.*?(?:at\s+least|minimum|min)\s+(\d+)",
                rf"(?:at\s+least|minimum|min)\s+(\d+).*?{segment_lower}\s+customers?",
            ],
            sentence,
            lambda m: constraints.segment_min_visits.update({segment_name: int(m.group(1))}),
        )

        set_if_found(
            [
                rf"{segment_lower}\s+customers?.*?(?:at\s+most|maximum|max|no\s+more\s+than)\s+(\d+)",
                rf"(?:at\s+most|maximum|max|no\s+more\s+than)\s+(\d+).*?{segment_lower}\s+customers?",
            ],
            sentence,
            lambda m: constraints.segment_max_visits.update({segment_name: int(m.group(1))}),
        )

        set_if_found(
            [
                rf"{segment_lower}\s+customers?.*?(?:exactly|only)\s+(\d+)",
                rf"(?:exactly|only)\s+(\d+).*?{segment_lower}\s+customers?",
            ],
            sentence,
            lambda m: (
                constraints.segment_min_visits.update({segment_name: int(m.group(1))}),
                constraints.segment_max_visits.update({segment_name: int(m.group(1))}),
            ),
        )


def parse_priority_and_preferences(sentence, constraints):
    set_if_found(
        [
            r"(?:high\s+priority\s+customers?.*?extra\s+bonus).*?(\d+)",
            r"(?:extra\s+high\s+bonus).*?(\d+)",
            r"(?:bonus\s+for\s+high\s+customers).*?(\d+)",
        ],
        sentence,
        lambda m: setattr(constraints, "extra_high_bonus", int(m.group(1))),
    )

    for segment_lower, segment_name in SEGMENT_MAP.items():
        set_if_found(
            [
                rf"{segment_lower}\s+(?:priority\s+weight|weight|score).*?(\d+)",
                rf"(?:set\s+)?{segment_lower}\s+priority\s+to\s+(\d+)",
            ],
            sentence,
            lambda m, s=segment_name: constraints.priority_weights.update({s: int(m.group(1))}),
        )

    preference_patterns = {
        "day_and_week": [r"(?:day\s+and\s+week\s+preference).*?(\d+)"],
        "day_only": [r"(?:day\s+preference|day\s+only\s+preference).*?(\d+)"],
        "week_only": [r"(?:week\s+preference|week\s+only\s+preference).*?(\d+)"],
        "other": [r"(?:other\s+preference|default\s+preference).*?(\d+)"],
    }

    for key, patterns in preference_patterns.items():
        set_if_found(
            patterns,
            sentence,
            lambda m, k=key: constraints.preference_weights.update({k: int(m.group(1))}),
        )


def parse_holidays(sentence, constraints):
    holidays = re.findall(r"\d{4}-\d{2}-\d{2}", sentence)

    if holidays and any(word in sentence for word in ["holiday", "off", "closed", "leave", "non working", "non-working"]):
        constraints.add_holidays.setdefault("ALL", [])
        constraints.add_holidays["ALL"].extend(holidays)


def parse_working_time(sentence, constraints):
    set_if_found(
        [
            r"(?:working\s+hours?|salesman\s+working\s+time|work\s+time|working\s+time)"
            r"\s*(?:from\s+)?"
            r"(\d{1,2})(?::(\d{2}))?\s*(am|pm)?"
            r"\s*(?:to|-)\s*"
            r"(\d{1,2})(?::(\d{2}))?\s*(am|pm)?"
        ],
        sentence,
        lambda m: constraints.salesman_working_hours.update(
            {
                "start_time": to_24h(m.group(1), m.group(2), m.group(3)),
                "end_time": to_24h(m.group(4), m.group(5), m.group(6)),
            }
        ),
    )


def parse_route_rules(sentence, constraints):
    set_if_found(
        [
            r"(?:maximum|max)\s+route\s+distance.*?(\d+(?:\.\d+)?)\s*km",
            r"(?:route\s+distance).*?(\d+(?:\.\d+)?)\s*km",
            r"(?:travel\s+distance).*?(\d+(?:\.\d+)?)\s*km",
        ],
        sentence,
        lambda m: setattr(constraints, "max_route_distance_km", float(m.group(1))),
    )

    set_if_found(
        [
            r"(?:maximum|max)\s+travel\s+time.*?(\d+)\s*(?:minutes|mins|min)",
            r"(?:travel\s+time).*?(\d+)\s*(?:minutes|mins|min)",
            r"(?:route\s+time).*?(\d+)\s*(?:minutes|mins|min)",
        ],
        sentence,
        lambda m: setattr(constraints, "max_travel_time_minutes", int(m.group(1))),
    )

    if any(
        phrase in sentence
        for phrase in [
            "nearby customers",
            "close customers",
            "closely located customers",
            "same route",
            "single route",
            "cluster customers",
            "group customers",
        ]
    ):
        constraints.group_nearby_customers = True

    set_if_found(
        [
            r"(?:nearby|close|closely located|cluster|group).*?within\s+(\d+(?:\.\d+)?)\s*km",
            r"(?:same route|single route).*?within\s+(\d+(?:\.\d+)?)\s*km",
        ],
        sentence,
        lambda m: (
            setattr(constraints, "group_nearby_customers", True),
            setattr(constraints, "nearby_customer_radius_km", float(m.group(1))),
        ),
    )


def validate_constraints(constraints):
    constraints.add_holidays = {
        k: sorted(list(set(v))) for k, v in constraints.add_holidays.items()
    }

    for segment in set(
        list(constraints.segment_min_visits.keys())
        + list(constraints.segment_max_visits.keys())
    ):
        min_v = constraints.segment_min_visits.get(segment)
        max_v = constraints.segment_max_visits.get(segment)

        if min_v is not None and max_v is not None and min_v > max_v:
            constraints.warnings.append(
                f"{segment} has min_visits greater than max_visits."
            )

    if (
        constraints.min_daily_visits is not None
        and constraints.territory_daily_capacity.get("ALL") is not None
        and constraints.min_daily_visits > constraints.territory_daily_capacity["ALL"]
    ):
        constraints.warnings.append(
            "min_daily_visits is greater than daily capacity."
        )

    if (
        constraints.max_route_distance_km is not None
        and constraints.max_route_distance_km <= 0
    ):
        constraints.warnings.append(
            "max_route_distance_km should be greater than 0."
        )

    if (
        constraints.max_travel_time_minutes is not None
        and constraints.max_travel_time_minutes <= 0
    ):
        constraints.warnings.append(
            "max_travel_time_minutes should be greater than 0."
        )

    return constraints


def parse_natural_language_constraints(user_request: str) -> NaturalLanguageConstraints:
    constraints = NaturalLanguageConstraints()
    constraints.explanation = "Parsed with modular rule-based parser."

    sentences = split_sentences(user_request)

    for sentence in sentences:
        parse_basic_capacity(sentence, constraints)
        parse_segment_rules(sentence, constraints)
        parse_priority_and_preferences(sentence, constraints)
        parse_holidays(sentence, constraints)
        parse_working_time(sentence, constraints)
        parse_route_rules(sentence, constraints)

    return validate_constraints(constraints)

In [7]:
test_requests = [
    """
    Set minimum daily visits to 4.
    High customers should be visited between 3 and 5 times.
    Medium customers should be visited at least 2 times.
    Give high priority customers extra bonus 40.
    """,

    """
    Daily capacity should be 8.
    Monthly capacity should be 150.
    Maximum customers should be 80.
    """,

    """
    No more than 7 visits per day.
    No more than 140 visits per month.
    Limit customers to 60.
    """,

    """
    Holiday on 2026-03-14.
    Closed on 2026-03-25.
    Salesman working time from 9am to 6pm.
    """,

    """
    Maximum route distance should be 42.5 km.
    Maximum travel time should be 90 minutes.
    Closely located customers within 3 km should be in a single route.
    """,

    """
    Low customers should be visited exactly 1 time.
    Medium customers should be visited between 2 and 3 times.
    High customers should be visited between 4 and 6 times.
    """,

    """
    Set high priority to 120.
    Medium priority weight should be 60.
    Low priority weight should be 15.
    Day preference should be 10.
    Week preference should be 5.
    Day and week preference should be 25.
    """,

    """
    Set minimum daily visits to 10.
    Daily capacity should be 6.
    High customers should be visited between 5 and 3 times.
    """
]

for i, request in enumerate(test_requests, start=1):
    print("=" * 100)
    print(f"TEST {i}")
    print("=" * 100)

    parsed = parse_natural_language_constraints(request)
    pprint(asdict(parsed))
    print()

TEST 1
{'add_holidays': {},
 'explanation': 'Parsed with modular rule-based parser.',
 'extra_high_bonus': 40,
 'group_nearby_customers': False,
 'max_route_distance_km': None,
 'max_travel_time_minutes': None,
 'min_daily_visits': 4,
 'nearby_customer_radius_km': None,
 'preference_weights': {},
 'priority_weights': {},
 'salesman_working_hours': {},
 'segment_max_visits': {'High': 5},
 'segment_min_visits': {'High': 3, 'Medium': 2},
 'territory_daily_capacity': {},
 'territory_max_customers_capacity': {},
 'territory_monthly_capacity': {},
 'warnings': []}

TEST 2
{'add_holidays': {},
 'explanation': 'Parsed with modular rule-based parser.',
 'extra_high_bonus': None,
 'group_nearby_customers': False,
 'max_route_distance_km': None,
 'max_travel_time_minutes': None,
 'min_daily_visits': None,
 'nearby_customer_radius_km': None,
 'preference_weights': {},
 'priority_weights': {},
 'salesman_working_hours': {},
 'segment_max_visits': {},
 'segment_min_visits': {},
 'territory_daily_cap

In [10]:
import re
from dataclasses import dataclass, field, asdict
from pprint import pprint


@dataclass
class NaturalLanguageConstraints:
    hard_constraints: dict = field(default_factory=dict)
    soft_constraints: dict = field(default_factory=dict)

    min_daily_visits: int | None = None
    extra_high_bonus: int | None = None

    priority_weights: dict = field(default_factory=dict)
    preference_weights: dict = field(default_factory=dict)

    segment_min_visits: dict = field(default_factory=dict)
    segment_max_visits: dict = field(default_factory=dict)

    territory_daily_capacity: dict = field(default_factory=dict)
    territory_monthly_capacity: dict = field(default_factory=dict)
    territory_max_customers_capacity: dict = field(default_factory=dict)
    territory_holidays: dict = field(default_factory=dict)

    salesperson_working_hours: dict = field(default_factory=dict)
    salesperson_daily_capacity: dict = field(default_factory=dict)
    salesperson_monthly_capacity: dict = field(default_factory=dict)
    salesperson_max_route_distance_km: dict = field(default_factory=dict)
    salesperson_max_travel_time_minutes: dict = field(default_factory=dict)

    day_daily_capacity: dict = field(default_factory=dict)
    day_preferred_segments: dict = field(default_factory=dict)
    day_blocked_segments: dict = field(default_factory=dict)

    add_holidays: dict = field(default_factory=dict)

    max_route_distance_km: float | None = None
    max_travel_time_minutes: int | None = None

    group_nearby_customers: bool = False
    nearby_customer_radius_km: float | None = None

    unparsed_instructions: list = field(default_factory=list)
    warnings: list = field(default_factory=list)
    explanation: str = ""


SEGMENT_MAP = {
    "high": "High",
    "medium": "Medium",
    "low": "Low",
}

DAYS = {
    "monday": "Monday",
    "tuesday": "Tuesday",
    "wednesday": "Wednesday",
    "thursday": "Thursday",
    "friday": "Friday",
    "saturday": "Saturday",
    "sunday": "Sunday",
}

HARD_WORDS = [
    "must",
    "mandatory",
    "strictly",
    "required",
    "cannot",
    "do not",
    "don't",
    "never",
    "only",
    "exactly",
    "at least",
    "at most",
    "no more than",
    "maximum",
    "minimum",
    "max",
    "min",
]

SOFT_WORDS = [
    "prefer",
    "preferred",
    "try",
    "if possible",
    "ideally",
    "prioritize",
    "give priority",
    "bonus",
]


def split_sentences(text: str) -> list[str]:
    text = text.replace("\n", ". ")
    text = re.sub(r"(\d)\.(\d)", r"\1<DECIMAL>\2", text)
    sentences = re.split(r"[.;]+", text)
    sentences = [s.replace("<DECIMAL>", ".") for s in sentences]
    return [s.strip().lower() for s in sentences if s.strip()]


def classify_strength(sentence: str) -> str:
    if any(word in sentence for word in SOFT_WORDS):
        return "soft"

    if any(word in sentence for word in HARD_WORDS):
        return "hard"

    return "hard"


def add_rule(constraints, strength, key, value):
    target = constraints.hard_constraints if strength == "hard" else constraints.soft_constraints
    target.setdefault(key, [])
    target[key].append(value)


def to_24h(hour, minute=None, ampm=None) -> str:
    hour = int(hour)
    minute = int(minute or 0)

    if ampm == "pm" and hour != 12:
        hour += 12

    if ampm == "am" and hour == 12:
        hour = 0

    return f"{hour:02d}:{minute:02d}"


def set_if_found(patterns, sentence, setter):
    for pattern in patterns:
        match = re.search(pattern, sentence)
        if match:
            setter(match)
            return True
    return False


def extract_territory(sentence: str) -> str | None:
    patterns = [
        r"(?:for|in)\s+([a-zA-Z][a-zA-Z\s_-]*?)\s+territory",
        r"([a-zA-Z][a-zA-Z\s_-]*?)\s+territory",
    ]

    blocked = {
        "daily",
        "monthly",
        "maximum",
        "max",
        "minimum",
        "min",
        "customer",
        "route",
        "salesperson",
        "salesman",
    }

    for pattern in patterns:
        match = re.search(pattern, sentence)
        if match:
            name = match.group(1).strip().title()
            if name.lower() not in blocked:
                return name

    return None


def extract_salesperson(sentence: str) -> str | None:
    patterns = [
        r"(?:salesperson|salesman|sales rep|rep)\s+([a-zA-Z][a-zA-Z_-]*)",
        r"^([a-zA-Z][a-zA-Z_-]*)\s+works\s+",
        r"^([a-zA-Z][a-zA-Z_-]*)\s+can\s+",
        r"^([a-zA-Z][a-zA-Z_-]*)\s+should\s+",
    ]

    blocked = {
        "daily",
        "monthly",
        "maximum",
        "minimum",
        "high",
        "medium",
        "low",
        "salesman",
        "salesperson",
    }

    for pattern in patterns:
        match = re.search(pattern, sentence)
        if match:
            name = match.group(1).strip().title()
            if name.lower() not in blocked:
                return name

    return None


def extract_day(sentence: str) -> str | None:
    for day_lower, day_name in DAYS.items():
        if day_lower in sentence:
            return day_name
    return None


def parse_basic_capacity(sentence, constraints):
    strength = classify_strength(sentence)
    parsed = False

    parsed |= set_if_found(
        [
            r"(?:minimum|min)\s+daily\s+visits?.*?(\d+)",
            r"(?:at\s+least)\s+(\d+)\s+visits?\s+per\s+day",
            r"(?:do|schedule)\s+minimum\s+(\d+)\s+visits?\s+daily",
        ],
        sentence,
        lambda m: (
            setattr(constraints, "min_daily_visits", int(m.group(1))),
            add_rule(constraints, strength, "min_daily_visits", int(m.group(1))),
        ),
    )

    parsed |= set_if_found(
        [
            r"(?:daily\s+capacity|daily\s+visit\s+capacity).*?(\d+)",
            r"(?:maximum|max)\s+(\d+)\s+visits?\s+per\s+day",
            r"(?:no\s+more\s+than)\s+(\d+)\s+visits?\s+per\s+day",
            r"(?:cap\s+daily\s+visits?\s+at)\s+(\d+)",
        ],
        sentence,
        lambda m: (
            constraints.territory_daily_capacity.update({"ALL": int(m.group(1))}),
            add_rule(constraints, strength, "daily_capacity", {"ALL": int(m.group(1))}),
        ),
    )

    parsed |= set_if_found(
        [
            r"(?:monthly\s+capacity|monthly\s+visit\s+capacity).*?(\d+)",
            r"(?:maximum|max)\s+(\d+)\s+visits?\s+per\s+month",
            r"(?:no\s+more\s+than)\s+(\d+)\s+visits?\s+per\s+month",
            r"(?:cap\s+monthly\s+visits?\s+at)\s+(\d+)",
        ],
        sentence,
        lambda m: (
            constraints.territory_monthly_capacity.update({"ALL": int(m.group(1))}),
            add_rule(constraints, strength, "monthly_capacity", {"ALL": int(m.group(1))}),
        ),
    )

    parsed |= set_if_found(
        [
            r"(?:maximum|max)\s+customers?.*?(\d+)",
            r"(?:customer\s+capacity).*?(\d+)",
            r"(?:select\s+only)\s+(\d+)\s+customers?",
            r"(?:limit\s+customers?\s+to)\s+(\d+)",
        ],
        sentence,
        lambda m: (
            constraints.territory_max_customers_capacity.update({"ALL": int(m.group(1))}),
            add_rule(constraints, strength, "max_customers_capacity", {"ALL": int(m.group(1))}),
        ),
    )

    return parsed


def parse_segment_rules(sentence, constraints):
    strength = classify_strength(sentence)
    parsed = False

    for segment_lower, segment_name in SEGMENT_MAP.items():
        if segment_lower not in sentence:
            continue

        matched = set_if_found(
            [
                rf"{segment_lower}\s+customers?.*?(?:between|from)\s+(\d+)\s+(?:and|to)\s+(\d+)",
                rf"{segment_lower}\s+customers?.*?(\d+)\s*-\s*(\d+)\s+visits?",
            ],
            sentence,
            lambda m: (
                constraints.segment_min_visits.update({segment_name: int(m.group(1))}),
                constraints.segment_max_visits.update({segment_name: int(m.group(2))}),
                add_rule(
                    constraints,
                    strength,
                    "segment_visit_range",
                    {
                        "segment": segment_name,
                        "min_visits": int(m.group(1)),
                        "max_visits": int(m.group(2)),
                    },
                ),
            ),
        )

        if matched:
            parsed = True
            continue

        parsed |= set_if_found(
            [
                rf"{segment_lower}\s+customers?.*?(?:at\s+least|minimum|min)\s+(\d+)",
                rf"(?:at\s+least|minimum|min)\s+(\d+).*?{segment_lower}\s+customers?",
            ],
            sentence,
            lambda m: (
                constraints.segment_min_visits.update({segment_name: int(m.group(1))}),
                add_rule(
                    constraints,
                    strength,
                    "segment_min_visits",
                    {"segment": segment_name, "min_visits": int(m.group(1))},
                ),
            ),
        )

        parsed |= set_if_found(
            [
                rf"{segment_lower}\s+customers?.*?(?:at\s+most|maximum|max|no\s+more\s+than)\s+(\d+)",
                rf"(?:at\s+most|maximum|max|no\s+more\s+than)\s+(\d+).*?{segment_lower}\s+customers?",
            ],
            sentence,
            lambda m: (
                constraints.segment_max_visits.update({segment_name: int(m.group(1))}),
                add_rule(
                    constraints,
                    strength,
                    "segment_max_visits",
                    {"segment": segment_name, "max_visits": int(m.group(1))},
                ),
            ),
        )

        parsed |= set_if_found(
            [
                rf"{segment_lower}\s+customers?.*?(?:exactly|only)\s+(\d+)",
                rf"(?:exactly|only)\s+(\d+).*?{segment_lower}\s+customers?",
            ],
            sentence,
            lambda m: (
                constraints.segment_min_visits.update({segment_name: int(m.group(1))}),
                constraints.segment_max_visits.update({segment_name: int(m.group(1))}),
                add_rule(
                    constraints,
                    strength,
                    "segment_exact_visits",
                    {"segment": segment_name, "visits": int(m.group(1))},
                ),
            ),
        )

    return parsed


def parse_priority_and_preferences(sentence, constraints):
    strength = classify_strength(sentence)
    parsed = False

    parsed |= set_if_found(
        [
            r"(?:high\s+priority\s+customers?.*?extra\s+bonus).*?(\d+)",
            r"(?:extra\s+high\s+bonus).*?(\d+)",
            r"(?:bonus\s+for\s+high\s+customers).*?(\d+)",
        ],
        sentence,
        lambda m: (
            setattr(constraints, "extra_high_bonus", int(m.group(1))),
            add_rule(constraints, "soft", "extra_high_bonus", int(m.group(1))),
        ),
    )

    for segment_lower, segment_name in SEGMENT_MAP.items():
        parsed |= set_if_found(
            [
                rf"{segment_lower}\s+(?:priority\s+weight|weight|score).*?(\d+)",
                rf"(?:set\s+)?{segment_lower}\s+priority\s+to\s+(\d+)",
            ],
            sentence,
            lambda m, s=segment_name: (
                constraints.priority_weights.update({s: int(m.group(1))}),
                add_rule(
                    constraints,
                    strength,
                    "priority_weight",
                    {"segment": s, "weight": int(m.group(1))},
                ),
            ),
        )

    preference_patterns = [
        ("day_and_week", [r"(?:day\s+and\s+week\s+preference).*?(\d+)"]),
        ("day_only", [r"(?:day\s+only\s+preference|day\s+preference).*?(\d+)"]),
        ("week_only", [r"(?:week\s+only\s+preference|^week\s+preference).*?(\d+)"]),
        ("other", [r"(?:other\s+preference|default\s+preference).*?(\d+)"]),
    ]

    for key, patterns in preference_patterns:
        parsed |= set_if_found(
            patterns,
            sentence,
            lambda m, k=key: (
                constraints.preference_weights.update({k: int(m.group(1))}),
                add_rule(
                    constraints,
                    "soft",
                    "preference_weight",
                    {"preference": k, "weight": int(m.group(1))},
                ),
            ),
        )

    return parsed


def parse_holidays(sentence, constraints):
    strength = classify_strength(sentence)
    territory = extract_territory(sentence) or "ALL"
    holidays = re.findall(r"\d{4}-\d{2}-\d{2}", sentence)

    if holidays and any(
        word in sentence
        for word in ["holiday", "off", "closed", "leave", "non working", "non-working"]
    ):
        constraints.add_holidays.setdefault(territory, [])
        constraints.add_holidays[territory].extend(holidays)

        constraints.territory_holidays.setdefault(territory, [])
        constraints.territory_holidays[territory].extend(holidays)

        add_rule(
            constraints,
            strength,
            "holidays",
            {"territory": territory, "dates": holidays},
        )

        return True

    return False


def parse_working_time(sentence, constraints):
    strength = classify_strength(sentence)
    salesperson = extract_salesperson(sentence) or "ALL"

    return set_if_found(
        [
            r"(?:working\s+hours?|salesman\s+working\s+time|salesperson\s+working\s+time|work\s+time|working\s+time|works)"
            r"\s*(?:from\s+)?"
            r"(\d{1,2})(?::(\d{2}))?\s*(am|pm)?"
            r"\s*(?:to|-)\s*"
            r"(\d{1,2})(?::(\d{2}))?\s*(am|pm)?"
        ],
        sentence,
        lambda m: (
            constraints.salesperson_working_hours.update(
                {
                    salesperson: {
                        "start_time": to_24h(m.group(1), m.group(2), m.group(3)),
                        "end_time": to_24h(m.group(4), m.group(5), m.group(6)),
                    }
                }
            ),
            add_rule(
                constraints,
                strength,
                "salesperson_working_hours",
                {
                    "salesperson": salesperson,
                    "start_time": to_24h(m.group(1), m.group(2), m.group(3)),
                    "end_time": to_24h(m.group(4), m.group(5), m.group(6)),
                },
            ),
        ),
    )


def parse_route_rules(sentence, constraints):
    strength = classify_strength(sentence)
    salesperson = extract_salesperson(sentence) or "ALL"
    parsed = False

    parsed |= set_if_found(
        [
            r".*route\s+distance.*?(\d+(?:\.\d+)?)\s*km",
            r".*travel\s+distance.*?(\d+(?:\.\d+)?)\s*km",
        ],
        sentence,
        lambda m: (
            setattr(constraints, "max_route_distance_km", float(m.group(1))),
            constraints.salesperson_max_route_distance_km.update(
                {salesperson: float(m.group(1))}
            ),
            add_rule(
                constraints,
                strength,
                "max_route_distance_km",
                {"salesperson": salesperson, "distance_km": float(m.group(1))},
            ),
        ),
    )

    parsed |= set_if_found(
        [
            r".*travel\s+time.*?(\d+)\s*(?:minutes|mins|min)",
            r".*route\s+time.*?(\d+)\s*(?:minutes|mins|min)",
        ],
        sentence,
        lambda m: (
            setattr(constraints, "max_travel_time_minutes", int(m.group(1))),
            constraints.salesperson_max_travel_time_minutes.update(
                {salesperson: int(m.group(1))}
            ),
            add_rule(
                constraints,
                strength,
                "max_travel_time_minutes",
                {"salesperson": salesperson, "minutes": int(m.group(1))},
            ),
        ),
    )

    if any(
        phrase in sentence
        for phrase in [
            "nearby customers",
            "close customers",
            "closely located customers",
            "same route",
            "single route",
            "cluster customers",
            "group customers",
        ]
    ):
        constraints.group_nearby_customers = True
        add_rule(constraints, strength, "group_nearby_customers", True)
        parsed = True

    parsed |= set_if_found(
        [
            r"(?:nearby|close|closely located|cluster|group).*?within\s+(\d+(?:\.\d+)?)\s*km",
            r"(?:same route|single route).*?within\s+(\d+(?:\.\d+)?)\s*km",
        ],
        sentence,
        lambda m: (
            setattr(constraints, "group_nearby_customers", True),
            setattr(constraints, "nearby_customer_radius_km", float(m.group(1))),
            add_rule(
                constraints,
                strength,
                "nearby_customer_radius_km",
                float(m.group(1)),
            ),
        ),
    )

    return parsed


def parse_territory_constraints(sentence, constraints):
    strength = classify_strength(sentence)
    territory = extract_territory(sentence)

    if not territory:
        return False

    parsed = False

    parsed |= set_if_found(
        [
            r"(?:daily\s+capacity|daily\s+visit\s+capacity).*?(\d+)",
            r"(?:maximum|max)\s+(\d+)\s+visits?\s+per\s+day",
        ],
        sentence,
        lambda m: (
            constraints.territory_daily_capacity.update({territory: int(m.group(1))}),
            add_rule(
                constraints,
                strength,
                "territory_daily_capacity",
                {"territory": territory, "capacity": int(m.group(1))},
            ),
        ),
    )

    parsed |= set_if_found(
        [
            r"(?:monthly\s+capacity|monthly\s+visit\s+capacity).*?(\d+)",
            r"(?:maximum|max)\s+(\d+)\s+visits?\s+per\s+month",
        ],
        sentence,
        lambda m: (
            constraints.territory_monthly_capacity.update({territory: int(m.group(1))}),
            add_rule(
                constraints,
                strength,
                "territory_monthly_capacity",
                {"territory": territory, "capacity": int(m.group(1))},
            ),
        ),
    )

    parsed |= set_if_found(
        [
            r"(?:maximum|max)\s+customers?.*?(\d+)",
            r"(?:customer\s+capacity).*?(\d+)",
        ],
        sentence,
        lambda m: (
            constraints.territory_max_customers_capacity.update(
                {territory: int(m.group(1))}
            ),
            add_rule(
                constraints,
                strength,
                "territory_max_customers_capacity",
                {"territory": territory, "capacity": int(m.group(1))},
            ),
        ),
    )

    return parsed


def parse_salesperson_constraints(sentence, constraints):
    strength = classify_strength(sentence)
    salesperson = extract_salesperson(sentence)

    if not salesperson:
        return False

    parsed = False

    parsed |= set_if_found(
        [
            r"(?:daily\s+capacity|daily\s+visit\s+capacity).*?(\d+)",
            r"(?:maximum|max)\s+(\d+)\s+visits?\s+per\s+day",
            r"(?:can\s+visit)\s+(\d+)\s+customers?\s+per\s+day",
        ],
        sentence,
        lambda m: (
            constraints.salesperson_daily_capacity.update(
                {salesperson: int(m.group(1))}
            ),
            add_rule(
                constraints,
                strength,
                "salesperson_daily_capacity",
                {"salesperson": salesperson, "capacity": int(m.group(1))},
            ),
        ),
    )

    parsed |= set_if_found(
        [
            r"(?:monthly\s+capacity|monthly\s+visit\s+capacity).*?(\d+)",
            r"(?:maximum|max)\s+(\d+)\s+visits?\s+per\s+month",
        ],
        sentence,
        lambda m: (
            constraints.salesperson_monthly_capacity.update(
                {salesperson: int(m.group(1))}
            ),
            add_rule(
                constraints,
                strength,
                "salesperson_monthly_capacity",
                {"salesperson": salesperson, "capacity": int(m.group(1))},
            ),
        ),
    )

    return parsed


def parse_day_constraints(sentence, constraints):
    strength = classify_strength(sentence)
    day = extract_day(sentence)

    if not day:
        return False

    parsed = False

    parsed |= set_if_found(
        [
            r"(?:max|maximum|no\s+more\s+than)\s+(\d+)\s+visits?",
            r".*?daily\s+capacity.*?(\d+)",
        ],
        sentence,
        lambda m: (
            constraints.day_daily_capacity.update({day: int(m.group(1))}),
            add_rule(
                constraints,
                strength,
                "day_daily_capacity",
                {"day": day, "capacity": int(m.group(1))},
            ),
        ),
    )

    for segment_lower, segment_name in SEGMENT_MAP.items():
        if segment_lower in sentence and any(
            word in sentence
            for word in ["prioritize", "prefer", "preferred", "give priority"]
        ):
            constraints.day_preferred_segments.setdefault(day, [])
            constraints.day_preferred_segments[day].append(segment_name)
            add_rule(
                constraints,
                "soft",
                "day_preferred_segment",
                {"day": day, "segment": segment_name},
            )
            parsed = True

        if segment_lower in sentence and any(
            word in sentence for word in ["avoid", "do not", "don't", "block", "no"]
        ):
            constraints.day_blocked_segments.setdefault(day, [])
            constraints.day_blocked_segments[day].append(segment_name)
            add_rule(
                constraints,
                "hard",
                "day_blocked_segment",
                {"day": day, "segment": segment_name},
            )
            parsed = True

    return parsed


def validate_constraints(constraints):
    constraints.add_holidays = {
        k: sorted(list(set(v))) for k, v in constraints.add_holidays.items()
    }

    constraints.territory_holidays = {
        k: sorted(list(set(v))) for k, v in constraints.territory_holidays.items()
    }

    constraints.day_preferred_segments = {
        k: sorted(list(set(v))) for k, v in constraints.day_preferred_segments.items()
    }

    constraints.day_blocked_segments = {
        k: sorted(list(set(v))) for k, v in constraints.day_blocked_segments.items()
    }

    for segment in set(
        list(constraints.segment_min_visits.keys())
        + list(constraints.segment_max_visits.keys())
    ):
        min_v = constraints.segment_min_visits.get(segment)
        max_v = constraints.segment_max_visits.get(segment)

        if min_v is not None and max_v is not None and min_v > max_v:
            constraints.warnings.append(
                f"{segment} has min_visits greater than max_visits."
            )

    if (
        constraints.min_daily_visits is not None
        and constraints.territory_daily_capacity.get("ALL") is not None
        and constraints.min_daily_visits > constraints.territory_daily_capacity["ALL"]
    ):
        constraints.warnings.append(
            "min_daily_visits is greater than daily capacity."
        )

    for day, preferred_segments in constraints.day_preferred_segments.items():
        blocked_segments = constraints.day_blocked_segments.get(day, [])

        for segment in preferred_segments:
            if segment in blocked_segments:
                constraints.warnings.append(
                    f"{segment} is both preferred and blocked on {day}."
                )

    return constraints


def parse_natural_language_constraints(user_request: str) -> NaturalLanguageConstraints:
    constraints = NaturalLanguageConstraints()
    constraints.explanation = "Parsed with rule parser. LLM can be used as fallback for unparsed instructions."

    sentences = split_sentences(user_request)

    for sentence in sentences:
        parsed = False

        parsed_territory = parse_territory_constraints(sentence, constraints)
        parsed_salesperson = parse_salesperson_constraints(sentence, constraints)

        parsed |= parsed_territory
        parsed |= parsed_salesperson
        parsed |= parse_day_constraints(sentence, constraints)

        if not parsed_territory and not parsed_salesperson:
            parsed |= parse_basic_capacity(sentence, constraints)

        parsed |= parse_segment_rules(sentence, constraints)
        parsed |= parse_priority_and_preferences(sentence, constraints)
        parsed |= parse_holidays(sentence, constraints)
        parsed |= parse_working_time(sentence, constraints)
        parsed |= parse_route_rules(sentence, constraints)

        if not parsed:
            constraints.unparsed_instructions.append(sentence)

    return validate_constraints(constraints)

In [11]:
test_requests = [
    """
    Must set minimum daily visits to 4.
    Prefer high customers between 3 and 5 times.
    Medium customers should be visited at least 2 times.
    Give high priority customers extra bonus 40.
    """,

    """
    Pune territory daily capacity should be 8.
    Mumbai territory monthly capacity should be 150.
    Delhi territory maximum customers should be 80.
    Mumbai territory holiday on 2026-03-14.
    """,

    """
    Salesperson Ravi works from 9am to 6pm.
    Salesperson Ravi max 7 visits per day.
    Salesperson Ravi maximum travel time 90 minutes.
    Salesperson Amit route distance should be 42.5 km.
    """,

    """
    Monday max 5 visits.
    Friday prioritize High customers.
    Saturday avoid Low customers.
    Sunday do not schedule Medium customers.
    """,

    """
    Closely located customers within 3 km should be in a single route.
    Avoid rainy areas.
    Customer should not be visited during lunch.
    """
]

for i, request in enumerate(test_requests, start=1):
    print("=" * 100)
    print(f"TEST {i}")
    print("=" * 100)

    parsed = parse_natural_language_constraints(request)
    pprint(asdict(parsed))
    print()

TEST 1
{'add_holidays': {},
 'day_blocked_segments': {},
 'day_daily_capacity': {},
 'day_preferred_segments': {},
 'explanation': 'Parsed with rule parser. LLM can be used as fallback for '
                'unparsed instructions.',
 'extra_high_bonus': 40,
 'group_nearby_customers': False,
 'hard_constraints': {'min_daily_visits': [4],
                      'segment_min_visits': [{'min_visits': 2,
                                              'segment': 'Medium'}]},
 'max_route_distance_km': None,
 'max_travel_time_minutes': None,
 'min_daily_visits': 4,
 'nearby_customer_radius_km': None,
 'preference_weights': {},
 'priority_weights': {},
 'salesperson_daily_capacity': {},
 'salesperson_max_route_distance_km': {},
 'salesperson_max_travel_time_minutes': {},
 'salesperson_monthly_capacity': {},
 'salesperson_working_hours': {},
 'segment_max_visits': {'High': 5},
 'segment_min_visits': {'High': 3, 'Medium': 2},
 'soft_constraints': {'extra_high_bonus': [40],
                      'se

In [17]:
!pip uninstall -y transformers accelerate tokenizers huggingface_hub safetensors
!pip install -q transformers==4.46.3 accelerate==1.1.1 tokenizers==0.20.3 huggingface_hub==0.26.5 safetensors

Found existing installation: transformers 5.9.0
Uninstalling transformers-5.9.0:
  Successfully uninstalled transformers-5.9.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: huggingface_hub 1.11.0
Uninstalling huggingface_hub-1.11.0:
  Successfully uninstalled huggingface_hub-1.11.0
Found existing installation: safetensors 0.7.0
Uninstalling safetensors-0.7.0:
  Successfully uninstalled safetensors-0.7.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [18]:
import transformers
import accelerate
import tokenizers
import huggingface_hub

print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("tokenizers:", tokenizers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

transformers: 5.0.0
accelerate: 1.13.0
tokenizers: 0.22.2
huggingface_hub: 1.11.0


In [1]:


import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [2]:
def qwen_parse_constraints_only(user_request):

    messages = [
        {
            "role": "system",
            "content": """
You are a JSON-only constraint parser for an OR-Tools field sales scheduling system.
Return ONLY a valid JSON array.
No markdown. No explanation.
"""
        },
        {
            "role": "user",
            "content": f"""
Extract scheduling constraints from this request.

Allowed output format:
[
  {{"type": "territory_daily_capacity", "territory": "Pune", "value": 8, "strength": "hard"}},
  {{"type": "salesperson_working_hours", "salesperson": "Ravi", "start_time": "09:00", "end_time": "18:00", "strength": "hard"}},
  {{"type": "salesperson_max_route_distance_km", "salesperson": "Amit", "value": 42.5, "strength": "hard"}},
  {{"type": "day_preferred_segment", "day": "Friday", "segment": "High", "strength": "soft"}},
  {{"type": "day_blocked_segment", "day": "Saturday", "segment": "Low", "strength": "hard"}},
  {{"type": "nearby_customer_radius_km", "value": 3, "strength": "hard"}},
  {{"type": "unknown", "text": "Avoid rainy areas"}}
]

Allowed types:
- min_daily_visits
- segment_min_visits
- segment_max_visits
- segment_exact_visits
- territory_daily_capacity
- territory_monthly_capacity
- territory_max_customers_capacity
- territory_holiday
- salesperson_working_hours
- salesperson_daily_capacity
- salesperson_monthly_capacity
- salesperson_max_route_distance_km
- salesperson_max_travel_time_minutes
- day_daily_capacity
- day_preferred_segment
- day_blocked_segment
- max_route_distance_km
- max_travel_time_minutes
- nearby_customer_radius_km
- extra_high_bonus
- priority_weight
- preference_weight
- unknown

Rules:
- Return JSON array only.
- Use double quotes.
- Use numbers for values.
- Use High, Medium, Low for segments.
- Use Monday, Tuesday, Wednesday, Thursday, Friday, Saturday, Sunday for days.
- Convert time to 24-hour HH:MM.
- strength is hard for must, max, min, exactly, at least, at most, avoid, do not.
- strength is soft for prefer, prioritize, bonus, if possible.
- Put unsupported instructions as type unknown.

Request:
{user_request}
"""
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    raw_output = tokenizer.decode(generated_ids, skip_special_tokens=True)

    parsed = extract_json_array(raw_output)

    if parsed is None:
        return {
            "parse_error": True,
            "raw_output": raw_output,
        }

    return parsed

In [3]:
import json
import re

def extract_json_array(text):
    text = text.strip()

    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return parsed
    except:
        pass

    match = re.search(r"\[.*\]", text, re.DOTALL)

    if match:
        try:
            parsed = json.loads(match.group())
            if isinstance(parsed, list):
                return parsed
        except:
            pass

    return None

In [6]:
user_request = """
Pune territory daily capacity should be 8.
Salesperson Ravi works from 9am to 6pm.
Salesperson Amit route distance should be 42.5 km.
Friday prioritize High customers.
Saturday avoid Low customers.
Closely located customers within 3 km should be in a single route.
Avoid rainy areas.
"""

parsed = qwen_parse_constraints_only(user_request)
print(parsed)

[{'type': 'territory_daily_capacity', 'territory': 'Pune', 'value': 8, 'strength': 'hard'}, {'type': 'salesperson_working_hours', 'salesperson': 'Ravi', 'start_time': '09:00', 'end_time': '18:00', 'strength': 'hard'}, {'type': 'salesperson_max_route_distance_km', 'salesperson': 'Amit', 'value': 42.5, 'strength': 'hard'}, {'type': 'day_preferred_segment', 'day': 'Friday', 'segment': 'High', 'strength': 'soft'}, {'type': 'day_blocked_segment', 'day': 'Saturday', 'segment': 'Low', 'strength': 'hard'}, {'type': 'nearby_customer_radius_km', 'value': 3, 'strength': 'hard'}]
